# PHY YOLO Pose Train

현재 수집 포맷의 port bbox와 4개 corner keypoint를 Ultralytics YOLO pose로 학습합니다.

- images: `images/<split>/<camera>/trial_<index>/*`
- annotations: `annotations/<split>/<camera>/trial_<index>/*.txt`
- metadata: `samples.jsonl`
- dataset config: `yolo_pose.yaml` (`kpt_shape: [4, 3]`)
- keypoint order: top-left, top-right, bottom-right, bottom-left

Class ID를 notebook에 고정하지 않고 `yolo_pose.yaml#names` 전체를 사용합니다. 따라서 SFP뿐 아니라 `sc_port` annotation이 추가되어도 같은 notebook으로 학습할 수 있습니다.

In [1]:
import json
import os
import random
from collections import Counter
from pathlib import Path

import yaml


def find_src_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "pixi.toml").is_file() and (path / "phy").is_dir():
            return path
        nested = path / "ws_aic" / "src"
        if (nested / "pixi.toml").is_file() and (nested / "phy").is_dir():
            return nested
    raise RuntimeError("ws_aic/src root를 찾지 못했습니다.")


SRC_ROOT = find_src_root(Path.cwd())
WS_ROOT = SRC_ROOT.parent
DATASET_DIR = WS_ROOT / "data" / "img2pos" / "phy_approach"
DATA_YAML = DATASET_DIR / "yolo_pose.yaml"
SAMPLES_JSONL = DATASET_DIR / "samples.jsonl"

MODEL_NAME = "yolo11s-pose.pt"
MODEL_ROOT = WS_ROOT / "model" / "phy_yolo_pose"
RUN_NAME = "board_view"

EPOCHS = 100
IMGSZ = 640
BATCH = 16
DEVICE = 0  # CPU: "cpu"
WORKERS = 8
PATIENCE = 20
IMAGE_EXTS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}

print(f"SRC_ROOT: {SRC_ROOT}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"DATA_YAML: {DATA_YAML}")

SRC_ROOT: /home/swlinux/Desktop/workspace/aic-physic/ws_aic/src
DATASET_DIR: /home/swlinux/Desktop/workspace/aic-physic/ws_aic/data/img2pos/phy_approach
DATA_YAML: /home/swlinux/Desktop/workspace/aic-physic/ws_aic/data/img2pos/phy_approach/yolo_pose.yaml


## Dataset config

저장소 루트 기준 상대경로 `data/img2pos/phy_approach`에서 수집기가 만든 `yolo_pose.yaml`을 읽습니다. 원본 `annotations`를 Ultralytics의 `labels` 규칙에 연결하는 학습용 상대 symlink view만 model directory에 만듭니다.

선택한 kernel에 패키지가 없다면 한 번만 실행하세요: `%pip install ultralytics PyYAML matplotlib`

In [2]:
if not DATA_YAML.is_file():
    raise FileNotFoundError(DATA_YAML)
if not SAMPLES_JSONL.is_file():
    raise FileNotFoundError(SAMPLES_JSONL)

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8")) or {}
raw_names = cfg.get("names", {})
if isinstance(raw_names, list):
    CLASS_NAMES = {index: str(name) for index, name in enumerate(raw_names)}
else:
    CLASS_NAMES = {int(index): str(name) for index, name in raw_names.items()}

KPT_COUNT, KPT_DIMS = map(int, cfg["kpt_shape"])
if (KPT_COUNT, KPT_DIMS) != (4, 3):
    raise ValueError(f"expected kpt_shape [4, 3], got {cfg['kpt_shape']}")
if not CLASS_NAMES:
    raise ValueError("yolo_pose.yaml#names is empty")

annotations_dir = DATASET_DIR / "annotations"
if not annotations_dir.is_dir():
    raise FileNotFoundError(annotations_dir)


def prepare_training_view() -> Path:
    view_dir = MODEL_ROOT / "dataset"
    view_dir.mkdir(parents=True, exist_ok=True)
    for name, target in {"images": DATASET_DIR / "images", "labels": annotations_dir}.items():
        link = view_dir / name
        relative_target = Path(os.path.relpath(target, link.parent))
        if link.is_symlink():
            if link.readlink() != relative_target:
                raise ValueError(f"unexpected symlink target: {link}")
        elif link.exists():
            raise FileExistsError(link)
        else:
            link.symlink_to(relative_target, target_is_directory=True)
    train_yaml = view_dir / "yolo_pose.yaml"
    train_cfg = dict(cfg)
    train_cfg.pop("path", None)
    train_yaml.write_text(
        yaml.safe_dump(train_cfg, sort_keys=False, allow_unicode=True),
        encoding="utf-8",
    )
    return train_yaml

print(f"classes: {CLASS_NAMES}")
print(f"kpt_shape: {[KPT_COUNT, KPT_DIMS]}")

classes: {0: 'SFP_00', 1: 'SFP_01', 2: 'SFP_10', 3: 'SFP_11', 4: 'SFP_20', 5: 'SFP_21', 6: 'SFP_30', 7: 'SFP_31', 8: 'SFP_40', 9: 'SFP_41', 10: 'sc_port'}
kpt_shape: [4, 3]


## Dataset validation

학습 전에 split 경로, image/annotation 1:1 대응, 17-field YOLO pose row, class ID, 정규화 좌표, visibility, `samples.jsonl` 참조를 검사합니다. 빈 annotation은 정상 negative sample입니다.

In [3]:
def split_dir(split: str) -> Path:
    value = Path(str(cfg[split]))
    if value.is_absolute():
        raise ValueError(f"{split} path must be relative: {value}")
    return DATASET_DIR / value


def iter_images(path: Path) -> list[Path]:
    return sorted(
        candidate
        for candidate in path.rglob("*")
        if candidate.is_file() and candidate.suffix.lower() in IMAGE_EXTS
    )


def annotation_for(image_path: Path) -> Path:
    relative = image_path.relative_to(DATASET_DIR / "images")
    return (annotations_dir / relative).with_suffix(".txt")


def validate_annotation(path: Path) -> list[str]:
    errors = []
    expected_fields = 5 + KPT_COUNT * KPT_DIMS
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        fields = line.split()
        if not fields:
            continue
        if len(fields) != expected_fields:
            errors.append(f"{path}:{line_number}: expected {expected_fields} fields, got {len(fields)}")
            continue
        try:
            class_id = int(fields[0])
            values = [float(value) for value in fields[1:]]
        except ValueError:
            errors.append(f"{path}:{line_number}: invalid number")
            continue
        if class_id not in CLASS_NAMES:
            errors.append(f"{path}:{line_number}: unknown class_id {class_id}")
        bbox = values[:4]
        if any(value < 0.0 or value > 1.0 for value in bbox):
            errors.append(f"{path}:{line_number}: bbox outside [0, 1]")
        if bbox[2] <= 0.0 or bbox[3] <= 0.0:
            errors.append(f"{path}:{line_number}: bbox width/height must be positive")
        for index in range(4, len(values), KPT_DIMS):
            x, y, visibility = values[index:index + 3]
            if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0):
                errors.append(f"{path}:{line_number}: keypoint outside [0, 1]")
            if visibility not in (0.0, 1.0, 2.0):
                errors.append(f"{path}:{line_number}: visibility must be 0, 1, or 2")
    return errors


errors = []
dataset_images = set()
split_counts = {}
for split in ("train", "val", "test"):
    if split not in cfg:
        continue
    image_root = split_dir(split)
    images = iter_images(image_root)
    split_counts[split] = len(images)
    dataset_images.update(path.relative_to(DATASET_DIR).as_posix() for path in images)
    if split in ("train", "val") and not images:
        errors.append(f"{split}: no images under {image_root}")
    expected_annotations = {annotation_for(path) for path in images}
    missing = [path for path in expected_annotations if not path.is_file()]
    annotation_root = annotations_dir / split
    actual_annotations = set(annotation_root.rglob("*.txt")) if annotation_root.exists() else set()
    orphan = actual_annotations - expected_annotations
    errors.extend(f"missing annotation: {path}" for path in sorted(missing))
    errors.extend(f"orphan annotation: {path}" for path in sorted(orphan))
    for annotation_path in sorted(actual_annotations):
        errors.extend(validate_annotation(annotation_path))

sample_images = set()
connector_counts = Counter()
camera_counts = Counter()
for line_number, line in enumerate(SAMPLES_JSONL.read_text(encoding="utf-8").splitlines(), 1):
    try:
        row = json.loads(line)
    except json.JSONDecodeError as exc:
        errors.append(f"samples.jsonl:{line_number}: {exc}")
        continue
    connector_counts[str(row.get("connector", "unknown"))] += 1
    for camera, relative_image in row.get("images", {}).items():
        sample_images.add(relative_image)
        camera_counts[camera] += 1
        if not (DATASET_DIR / relative_image).is_file():
            errors.append(f"samples.jsonl:{line_number}: missing image {relative_image}")
        relative_annotation = row.get("annotations", {}).get(camera)
        if relative_annotation is None or not (DATASET_DIR / relative_annotation).is_file():
            errors.append(f"samples.jsonl:{line_number}: missing annotation for {camera}")

errors.extend(f"image missing from samples.jsonl: {path}" for path in sorted(dataset_images - sample_images))
errors.extend(f"samples.jsonl references unknown image: {path}" for path in sorted(sample_images - dataset_images))

print(f"split images: {split_counts}")
print(f"connectors: {dict(connector_counts)}")
print(f"cameras: {dict(camera_counts)}")
print(f"classes: {CLASS_NAMES}")
if errors:
    preview = "\n".join(errors[:100])
    raise ValueError(f"Dataset validation failed with {len(errors)} errors.\n{preview}")
print(f"OK: images={len(dataset_images)}, expected label fields={5 + KPT_COUNT * KPT_DIMS}")

split images: {'train': 6219, 'val': 1362, 'test': 1337}
connectors: {'SFP': 2973}
cameras: {'left': 2973, 'center': 2972, 'right': 2973}
classes: {0: 'SFP_00', 1: 'SFP_01', 2: 'SFP_10', 3: 'SFP_11', 4: 'SFP_20', 5: 'SFP_21', 6: 'SFP_30', 7: 'SFP_31', 8: 'SFP_40', 9: 'SFP_41', 10: 'sc_port'}
OK: images=8918, expected label fields=17


## Train

`classes` 인자를 넘기지 않으므로 YAML에 선언된 SFP와 SC class를 모두 학습합니다. Corner 의미 보존을 위해 horizontal/vertical flip은 비활성화합니다.

In [4]:
from ultralytics import YOLO

TRAIN_YAML = prepare_training_view()
model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(TRAIN_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(MODEL_ROOT),
    name=RUN_NAME,
    patience=PATIENCE,
    save=True,
    save_period=10,
    plots=True,
    device=DEVICE,
    workers=WORKERS,
    fliplr=0.0,
    flipud=0.0,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
print(f"run: {RUN_DIR}")
print(f"best: {BEST_PT}")

WARNING ⚠️ Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at '/home/swlinux/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.117 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.114 🚀 Python-3.11.9 torch-2.12.1+cu130 CUDA:0 (NVIDIA GeForce GTX 1050, 4034MiB)


/home/swlinux/Desktop/workspace/CJ-Logistics-Challenge/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:384: UserWarning: Found GPU0 NVIDIA GeForce GTX 1050 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/swlinux/Desktop/workspace/CJ-Logistics-Challenge/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:502: UserWarning: 
NVIDIA GeForce GTX 1050 with CUDA capability 

engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/swlinux/Desktop/workspace/aic-physic/ws_aic/model/phy_yolo_pose/dataset/yolo_pose.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=board_view, nbs=64, nms=False, opset=None, opt

AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Resume or validate

필요한 셀만 주석을 해제해 실행합니다.

In [ ]:
# 완전 재개
# model = YOLO(str(LAST_PT))
# train_results = model.train(resume=True)

# 학습 없이 validation
# model = YOLO(str(BEST_PT))
# metrics = model.val(
#     data=str(TRAIN_YAML), imgsz=IMGSZ, batch=BATCH, device=DEVICE
# )
# print(metrics)

## Prediction preview

In [ ]:
import matplotlib.pyplot as plt

preview_images = iter_images(split_dir("val")) or iter_images(split_dir("train"))
sample = random.choice(preview_images)
model = YOLO(str(BEST_PT))
prediction = model.predict(
    source=str(sample),
    imgsz=IMGSZ,
    device=DEVICE,
    save=True,
    project=str(MODEL_ROOT),
    name=f"{RUN_NAME}_predict",
)
annotated_bgr = prediction[0].plot()
plt.figure(figsize=(10, 8))
plt.imshow(annotated_bgr[..., ::-1])
plt.axis("off")
plt.title(sample.name)
print(sample)